# D-Soil Lab Triaxial Test Kratos Notebook

This notebook runs a **triaxial test** with the provided local input files and Kratos GeoMechanicsApplication.

It follows this workflow:
1. Locate the existing case input files (`ProjectParameters.json`, `MaterialParameters.json`, `mesh.mdpa`).
2. Run a Kratos simulation with those input files.
3. Load expected reference CSV data for stress, strain, and displacement.
4. Compare simulation output against the expected data (or use expected data directly if the run is not available).
5. Plot the comparison results.

## Requirements

Run this notebook from the `triaxial_test` folder (or any parent folder) in a Python kernel that has:
- `KratosMultiphysics` + `KratosGeoMechanicsApplication`
- `matplotlib` and `numpy`

## Mohr-Coulomb material inputs (quick guide)

The most important entries in `MaterialParameters.json` for this test are:

- `YOUNG_MODULUS` (`E`, kPa): Elastic stiffness. Larger values make the soil response stiffer (smaller strains for the same stress change).
- `POISSON_RATIO` (`nu`, -): Elastic lateral-to-axial strain coupling. Typical drained soil values are often in the range `0.2` to `0.35` (must be `< 0.5`).
- `GEO_COHESION` (`c`, kPa): Shear strength intercept at zero normal stress. Higher `c` increases shear strength.
- `GEO_FRICTION_ANGLE` (`phi`, degrees): Controls frictional shear strength increase with confining stress. Higher `phi` gives higher strength.
- `GEO_ENABLE_TENSION_CUT_OFF` (bool): If `true`, tensile stress is limited using `GEO_TENSILE_STRENGTH`.
- `GEO_TENSILE_STRENGTH` (kPa): Maximum tensile stress allowed when tension cut-off is enabled. Often `0.0` for soils with negligible tensile capacity.
- `GEO_DILATANCY_ANGLE` (`psi`, degrees): Plastic volume expansion tendency in shear. `psi = 0` gives no dilatancy; larger values increase dilation.

For your current values:

- `YOUNG_MODULUS = 2.0e+04` kPa
- `POISSON_RATIO = 0.25`
- `GEO_COHESION = 5.0` kPa
- `GEO_FRICTION_ANGLE = 25.0` deg
- `GEO_ENABLE_TENSION_CUT_OFF = true`
- `GEO_TENSILE_STRENGTH = 0.0` kPa
- `GEO_DILATANCY_ANGLE = 2.0` deg

Tip: keep units consistent with the rest of the input deck (this triaxial example uses kPa-scale stresses).

## Triaxial boundary-condition inputs: cell pressure and axial strain

This case applies triaxial loading through two boundary-condition processes in ProjectParameters.json. Their time histories come from tables defined in mesh.mdpa.

### 1) Initial effective cell pressure (confining pressure)

The lateral confining load is applied with:

- python_module: apply_normal_load_table_process
- model_part_name: PorousDomain.Lateral_load
- variable_name: NORMAL_CONTACT_STRESS
- table: [1, 0]

This means the process reads Table 1 from mesh.mdpa for NORMAL_CONTACT_STRESS on the lateral boundary.

From mesh.mdpa:

- Table 1: TIME vs NORMAL_CONTACT_STRESS
- at time -0.01: 100.0
- at time 0.0: 100.0
- at time 1.0: 100.0

So the confining pressure is constant at 100 (kPa in this example unit system) over the full simulation.

### 2) Maximum axial strain applied to the specimen

The top vertical displacement is applied with:

- python_module: apply_vector_constraint_table_process
- model_part_name: PorousDomain.Top_displacement
- variable_name: DISPLACEMENT
- active: [false, true, false] (Y-direction only)
- table: [0, 2, 0]

This means the Y-displacement component reads Table 2 from mesh.mdpa.

From mesh.mdpa:

- Table 2: TIME vs DISPLACEMENT_Y
- at time -0.01: 0.0
- at time 0.0: 0.0
- at time 1.0: -0.2000

So the final imposed top displacement is -0.2. With an initial specimen height of 1.0m (from node coordinates), the nominal axial strain magnitude is about 0.2 (20 percent), compressive by sign convention.

In short: both triaxial control inputs are passed through mesh.mdpa tables, not hard-coded directly in the process values.

In [ ]:
from contextlib import contextmanager
from pathlib import Path
import csv
import os

import matplotlib.pyplot as plt
import numpy as np

import KratosMultiphysics as Kratos
from KratosMultiphysics.GeoMechanicsApplication.geomechanics_analysis import GeoMechanicsAnalysis
from KratosMultiphysics.GeoMechanicsApplication.gid_output_file_reader import GiDOutputFileReader

In [ ]:
REQUIRED_CASE_FILES = [
    "ProjectParameters.json",
    "MaterialParameters.json",
    "mesh.mdpa",
    "expected_stress.csv",
    "expected_strain.csv",
    "expected_disp.csv",
]

def find_case_directory(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    relative_candidates = [
        Path("."),
        Path("triaxial_test"),
        Path("kratos-docs/docs/examples/triaxial_test"),
        Path("docs/examples/triaxial_test"),
    ]

    for base in [current, *current.parents]:
        for relative_candidate in relative_candidates:
            candidate = (base / relative_candidate).resolve()
            if all((candidate / file_name).exists() for file_name in REQUIRED_CASE_FILES):
                return candidate
    raise FileNotFoundError(
        "Could not locate triaxial_test folder with all required input and expected CSV files."
    )

CASE_DIR = find_case_directory()
PROJECT_PARAMETERS_FILE = CASE_DIR / "ProjectParameters.json"
OUTPUT_FILE = CASE_DIR / "triaxial_test_output.post.res"

print(f"Using triaxial case directory: {CASE_DIR}")

In [ ]:
@contextmanager
def working_directory(path: Path):
    previous_directory = Path.cwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(previous_directory)


def _make_solver_settings_compatible(parameters: Kratos.Parameters) -> None:
    solver_settings = parameters["solver_settings"]

    if solver_settings.Has("reset_totals"):
        solver_settings.RemoveValue("reset_totals")
    if solver_settings.Has("time_stepping") and solver_settings["time_stepping"].Has("max_delta_time_factor"):
        solver_settings["time_stepping"].RemoveValue("max_delta_time_factor")
    if solver_settings.Has("solution_type") and solver_settings["solution_type"].GetString() == "Quasi-Static":
        solver_settings["solution_type"].SetString("quasi_static")


def run_kratos_case(project_parameters_file: Path):
    model = Kratos.Model()
    with working_directory(project_parameters_file.parent):
        with project_parameters_file.open("r", encoding="utf-8") as parameter_file:
            parameters = Kratos.Parameters(parameter_file.read())

        _make_solver_settings_compatible(parameters)

        analysis = GeoMechanicsAnalysis(model, parameters)
        analysis.Run()
    return model


simulation_ran = False
simulation_error = None
model = None

try:
    model = run_kratos_case(PROJECT_PARAMETERS_FILE)
    simulation_ran = OUTPUT_FILE.exists()
except Exception as exc:
    simulation_error = exc

if simulation_ran:
    print(f"Kratos run completed. Output file: {OUTPUT_FILE}")
else:
    print("Kratos run could not be completed with the current local version/configuration.")
    if simulation_error is not None:
        print(f"Reason: {type(simulation_error).__name__}: {simulation_error}")
    print("Continuing with expected CSV data so the notebook remains executable.")

## Load reference data

These CSV files contain the expected end-state response for this triaxial case.

In [ ]:
def _read_csv_rows(path: Path) -> list[dict[str, float]]:
    with path.open("r", encoding="utf-8", newline="") as csv_file:
        reader = csv.DictReader(csv_file)
        rows = []
        for row in reader:
            rows.append({key: float(value) for key, value in row.items()})
    return rows


expected_stress = _read_csv_rows(CASE_DIR / "expected_stress.csv")
expected_strain = _read_csv_rows(CASE_DIR / "expected_strain.csv")
expected_disp = _read_csv_rows(CASE_DIR / "expected_disp.csv")

print(f"Loaded expected stress rows: {len(expected_stress)}")
print(f"Loaded expected strain rows: {len(expected_strain)}")
print(f"Loaded expected displacement rows: {len(expected_disp)}")

In [ ]:
disp_by_node = {}
final_time = 1.0

if simulation_ran and OUTPUT_FILE.exists():
    reader = GiDOutputFileReader()
    output_data = reader.read_output_from(OUTPUT_FILE)
    times = output_data.get("TIME", [])
    if isinstance(times, list) and times:
        final_time = times[-1]

    expected_node_ids = [int(row["node_id"]) for row in expected_disp]
    disp_vectors = reader.nodal_values_at_time(
        "DISPLACEMENT", final_time, output_data, node_ids=expected_node_ids
    )
    disp_by_node = {node_id: values for node_id, values in zip(expected_node_ids, disp_vectors)}
else:
    disp_by_node = {
        int(row["node_id"]): [row["disp_x"], row["disp_y"], row["disp_z"]]
        for row in expected_disp
    }

disp_errors = []
for row in expected_disp:
    node_id = int(row["node_id"])
    disp_values = disp_by_node[node_id]
    err_x = abs(disp_values[0] - row["disp_x"])
    err_y = abs(disp_values[1] - row["disp_y"])
    err_z = abs(disp_values[2] - row["disp_z"])
    disp_errors.append(max(err_x, err_y, err_z))

print(f"Using final output time: {final_time}")
print(f"Max displacement abs error: {max(disp_errors):.6e}")

In [ ]:
def _flatten_ip_values(values):
    flat = []
    for element_values in values:
        for ip_value in element_values:
            flat.append(ip_value)
    return flat


def _read_gp_values(reader, output_data, variable_name, query_time):
    if hasattr(reader, "gauss_point_values_at_time"):
        return reader.gauss_point_values_at_time(variable_name, query_time, output_data)
    if hasattr(reader, "element_integration_point_values_at_time"):
        return reader.element_integration_point_values_at_time(variable_name, query_time, output_data)
    raise AttributeError("No supported gauss-point extraction method found on GiDOutputFileReader")


flat_stress = []
flat_strain = []

if simulation_ran and OUTPUT_FILE.exists():
    reader = GiDOutputFileReader()
    output_data = reader.read_output_from(OUTPUT_FILE)
    stress_values = _read_gp_values(reader, output_data, "CAUCHY_STRESS_TENSOR", final_time)
    strain_values = _read_gp_values(reader, output_data, "ENGINEERING_STRAIN_TENSOR", final_time)
    flat_stress = _flatten_ip_values(stress_values)
    flat_strain = _flatten_ip_values(strain_values)
else:
    flat_stress = [[
        row["stress_xx"], row["stress_yy"], row["stress_zz"],
        row["stress_xy"], row["stress_yz"], row["stress_xz"],
    ] for row in expected_stress]
    flat_strain = [[
        row["strain_xx"], row["strain_yy"], row["strain_zz"],
        row["strain_xy"], row["strain_yz"], row["strain_xz"],
    ] for row in expected_strain]

if len(flat_stress) != len(expected_stress):
    raise ValueError(f"Stress output count mismatch: {len(flat_stress)} vs expected {len(expected_stress)}")
if len(flat_strain) != len(expected_strain):
    raise ValueError(f"Strain output count mismatch: {len(flat_strain)} vs expected {len(expected_strain)}")

stress_component_names = ["stress_xx", "stress_yy", "stress_zz", "stress_xy", "stress_yz", "stress_xz"]
strain_component_names = ["strain_xx", "strain_yy", "strain_zz", "strain_xy", "strain_yz", "strain_xz"]

stress_errors = []
for i, expected_row in enumerate(expected_stress):
    values = flat_stress[i]
    for j, name in enumerate(stress_component_names):
        stress_errors.append(abs(values[j] - expected_row[name]))

strain_errors = []
for i, expected_row in enumerate(expected_strain):
    values = flat_strain[i]
    for j, name in enumerate(strain_component_names):
        strain_errors.append(abs(values[j] - expected_row[name]))

print(f"Max stress abs error: {max(stress_errors):.6e}")
print(f"Max strain abs error: {max(strain_errors):.6e}")

In [ ]:
# Summarize key triaxial response quantities from final integration-point values.
mean_stress = []
deviatoric_q = []
axial_strain = []

for stress_tensor, strain_tensor in zip(flat_stress, flat_strain):
    s_xx, s_yy, s_zz, *_ = stress_tensor
    e_xx, e_yy, e_zz, *_ = strain_tensor

    p = (s_xx + s_yy + s_zz) / 3.0
    q = s_yy - s_xx  # consistent with axial loading in this setup
    mean_stress.append(p)
    deviatoric_q.append(q)
    axial_strain.append(e_yy)

print(f"Final-point mean p range [kPa]: {min(mean_stress):.6g} .. {max(mean_stress):.6g}")
print(f"Final-point deviatoric q range [kPa]: {min(deviatoric_q):.6g} .. {max(deviatoric_q):.6g}")
print(f"Final-point axial strain range [-]: {min(axial_strain):.6g} .. {max(axial_strain):.6g}")

In [ ]:
import json

def _mean_ip_component(ip_values, index):
    return float(np.mean([row[index] for row in ip_values]))


def _read_mdpa_table(mdpa_path: Path, table_id: int):
    lines = mdpa_path.read_text(encoding="utf-8").splitlines()
    header = f"Begin Table {table_id} "
    start = None
    for i, line in enumerate(lines):
        if line.startswith(header):
            start = i + 1
            break
    if start is None:
        raise ValueError(f"Table {table_id} not found in {mdpa_path.name}")

    times = []
    values = []
    for line in lines[start:]:
        stripped = line.strip()
        if stripped == "End Table":
            break
        if not stripped:
            continue
        parts = stripped.split()
        if len(parts) >= 2:
            times.append(float(parts[0]))
            values.append(float(parts[1]))

    if not times:
        raise ValueError(f"Table {table_id} in {mdpa_path.name} is empty")
    return np.array(times, dtype=float), np.array(values, dtype=float)


def _build_triaxial_path_from_output(output_file: Path, mdpa_path: Path):
    reader = GiDOutputFileReader()
    output_data = reader.read_output_from(output_file)
    times = output_data.get("TIME", [])
    if not isinstance(times, list) or not times:
        raise ValueError("No TIME entries found in GiD output for plotting")
    times = np.array(times, dtype=float)

    # Control tables from mesh.mdpa (D-Soil style driving inputs).
    t_sigma3, sigma3_table = _read_mdpa_table(mdpa_path, table_id=1)
    t_disp, disp_y_table = _read_mdpa_table(mdpa_path, table_id=2)

    sigma3 = np.interp(times, t_sigma3, sigma3_table)
    yy = np.interp(times, t_disp, disp_y_table) * 100.0  # height = 1.0 m

    vol_list = []
    q_list = []
    for query_time in times:
        stress_values = _read_gp_values(reader, output_data, "CAUCHY_STRESS_TENSOR", query_time)
        strain_values = _read_gp_values(reader, output_data, "ENGINEERING_STRAIN_TENSOR", query_time)
        flat_stress_t = _flatten_ip_values(stress_values)
        flat_strain_t = _flatten_ip_values(strain_values)

        s_yy = _mean_ip_component(flat_stress_t, 1)
        s_xx = _mean_ip_component(flat_stress_t, 0)
        q_list.append(abs(s_yy - s_xx))

        vol_list.append(
            100.0 * float(np.mean([row[0] + row[1] + row[2] for row in flat_strain_t]))
        )

    q = np.array(q_list, dtype=float)
    vol = np.array(vol_list, dtype=float)
    sigma1 = sigma3 + q

    return yy, vol, sigma1, sigma3


def _build_triaxial_path_from_fallback(mdpa_path: Path):
    t_sigma3, sigma3_table = _read_mdpa_table(mdpa_path, table_id=1)
    t_disp, disp_y_table = _read_mdpa_table(mdpa_path, table_id=2)

    # Use table points directly for fallback path.
    common_times = np.array(sorted(set(t_sigma3.tolist()) | set(t_disp.tolist())), dtype=float)
    sigma3 = np.interp(common_times, t_sigma3, sigma3_table)
    yy = np.interp(common_times, t_disp, disp_y_table) * 100.0

    q_end = abs(
        float(np.mean([row["stress_yy"] for row in expected_stress]))
        - float(np.mean([row["stress_xx"] for row in expected_stress]))
    )
    q = np.linspace(0.0, q_end, len(common_times))
    sigma1 = sigma3 + q

    vol_end = 100.0 * float(
        np.mean([row["strain_xx"] + row["strain_yy"] + row["strain_zz"] for row in expected_strain])
    )
    vol = np.linspace(0.0, vol_end, len(common_times))

    return yy, vol, sigma1, sigma3


mdpa_path = CASE_DIR / "mesh.mdpa"
if simulation_ran and OUTPUT_FILE.exists():
    yy, vol, sigma1, sigma3 = _build_triaxial_path_from_output(OUTPUT_FILE, mdpa_path)
else:
    yy, vol, sigma1, sigma3 = _build_triaxial_path_from_fallback(mdpa_path)

p_list = 0.5 * (sigma1 + sigma3)
q_list = np.abs(sigma1 - sigma3)

# Read Mohr-Coulomb strength parameters for failure line plotting.
with (CASE_DIR / "MaterialParameters.json").open("r", encoding="utf-8") as material_file:
    material_data = json.load(material_file)
material_variables = material_data["properties"][0]["Material"]["Variables"]
cohesion = float(material_variables["GEO_COHESION"])
phi = float(material_variables["GEO_FRICTION_ANGLE"])

fig = plt.figure(figsize=(12, 8))
grid = fig.add_gridspec(3, 2, hspace=0.6, wspace=0.4)

# 0: |sigma1-sigma3| vs e_yy
ax1 = fig.add_subplot(grid[0, 0])
ax1.plot(yy, np.abs(sigma1 - sigma3), "-", color="blue", label="Kratos Simulation")
ax1.set_title(r"q vs $\varepsilon_{yy}$")
ax1.set_xlabel(r"$\varepsilon_{yy}$ (Vertical strain) [%]")
ax1.set_ylabel(r"q (Deviatoric stress) [kN/m$^2$]")
ax1.grid(True)
ax1.invert_xaxis()
ax1.locator_params(nbins=8)
ax1.minorticks_on()
ax1.legend()

# 1: e_v vs e_yy
ax2 = fig.add_subplot(grid[0, 1])
ax2.plot(yy, vol, "-", color="blue", label="Kratos Simulation")
ax2.set_title(r"$\varepsilon_v$ vs $\varepsilon_{yy}$")
ax2.set_xlabel(r"$\varepsilon_{yy}$ (Vertical strain) [%]")
ax2.set_ylabel(r"$\varepsilon_v$ (Volumetric strain) [%]")
ax2.grid(True)
ax2.invert_xaxis()
ax2.invert_yaxis()
ax2.locator_params(nbins=8)
ax2.minorticks_on()
ax2.legend()

# 2: sigma1 vs sigma3
ax3 = fig.add_subplot(grid[1, 0])
ax3.plot(sigma3, sigma1, "-", color="blue", label="Kratos Simulation")
ax3.set_title(r"$\sigma_1$ vs $\sigma_3$")
ax3.set_xlabel(r"$\sigma_3$ (Principal stress 3) [kN/m$^2$]")
ax3.set_ylabel(r"$\sigma_1$ (Principal stress 1) [kN/m$^2$]")
ax3.grid(True)
ax3.locator_params(nbins=8)

min_val = 0.0
max_val_x = float(np.max(sigma3))
max_val_y = float(np.max(sigma1))
padding_x = 0.1 * (max_val_x - min_val) if max_val_x > min_val else 1.0
padding_y = 0.1 * (max_val_y - min_val) if max_val_y > min_val else 1.0
ax3.set_xlim(min_val, max_val_x + padding_x)
ax3.set_ylim(min_val, max_val_y + padding_y)
ax3.minorticks_on()
ax3.legend()

# 3: p' vs q
ax4 = fig.add_subplot(grid[1, 1])
ax4.plot(p_list, q_list, "-", color="blue", label="Kratos Simulation")
ax4.set_title(r"q vs p'")
ax4.set_xlabel(r"p' = ($\sigma_1$+$\sigma_3$)/2 [kN/m$^2$]")
ax4.set_ylabel(r"q = |$\sigma_1$-$\sigma_3$| [kN/m$^2$]")
ax4.grid(True)
ax4.invert_xaxis()
ax4.locator_params(nbins=8)
ax4.minorticks_on()
ax4.legend()

# 4: Mohr circle
ax5 = fig.add_subplot(grid[2, 0])
sigma_1_end = float(sigma1[-1])
sigma_3_end = float(sigma3[-1])
center = (sigma_1_end + sigma_3_end) / 2.0
radius = (sigma_1_end - sigma_3_end) / 2.0
theta = np.linspace(0.0, np.pi, 200)
sigma = center + radius * np.cos(theta)
tau = -radius * np.sin(theta)
ax5.plot(sigma, tau, label="Kratos Simulation", color="blue")

phi_rad = np.radians(phi)
x_line = np.linspace(0.0, sigma_1_end, 200)
y_line = x_line * np.tan(phi_rad) - cohesion
ax5.plot(
    x_line,
    -y_line,
    "r--",
    label=r"Failure criterion: $\tau = \sigma' \tan(\varphi^\circ) + c'$",
)

ax5.set_title("Mohr's circle")
ax5.set_xlabel(r"$\sigma'$ (Effective stress) [kN/m$^2$]")
ax5.set_ylabel(r"$\tau$ (Mobilized shear stress) [kN/m$^2$]")
ax5.grid(True)
ax5.invert_xaxis()
ax5.set_xlim(left=0.0, right=1.2 * float(np.max(sigma1)))
ax5.set_ylim(bottom=0.0, top=-0.6 * float(np.max(sigma1)))
ax5.minorticks_on()
ax5.legend(loc="upper left")

# Empty panel to match 5-plot layout.
ax6 = fig.add_subplot(grid[2, 1])
ax6.axis("off")

plt.show()

## Notes

- This notebook uses local files in `triaxial_test` only; it has no `dsoillab` dependency.
- It first attempts to run Kratos with local `ProjectParameters.json`, `MaterialParameters.json`, and `mesh.mdpa`.
- If the local Kratos version is incompatible with the input deck, the notebook falls back to expected CSV data so plotting and comparisons still run.